# Use Case 4 - ETL for Monitoring and Model Review on AWS

This lab treats monitoring as another **ETL-style batch workflow**.


## What this lab covers
- Amazon S3 monitoring input zone
- Baseline profiling and incoming batch comparison
- Bias or explainability discussion in SageMaker
- Decision ladder: retain, review, retrain, rollback


## ETL flow in one line
`Extract baseline and incoming data -> Transform into comparable profiles -> Load findings and review artifacts -> Decide lifecycle action`


In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
BASE = Path("..")
train_df = pd.read_csv(BASE / "04_Datasets" / "baseline" / "train.csv")
future_df = pd.read_csv(BASE / "04_Datasets" / "monitoring" / "future_scoring_sample.csv")
print(train_df.shape, future_df.shape)


## 1. Extract the baseline and incoming batch
In the live AWS demo, show the baseline in `churn/ml_ready/` and the incoming batch in `churn/monitoring/incoming/`.


## 2. Transform into monitoring profiles


In [ ]:
baseline_rows = []
for col in train_df.columns:
    if pd.api.types.is_numeric_dtype(train_df[col]):
        baseline_rows.append({"column_name": col, "column_type": "numeric", "mean": train_df[col].mean(), "std": train_df[col].std(), "null_rate": train_df[col].isna().mean()})
    else:
        baseline_rows.append({"column_name": col, "column_type": "categorical", "top_value": train_df[col].astype(str).fillna("MISSING").value_counts().index[0], "null_rate": train_df[col].isna().mean()})
baseline_profile = pd.DataFrame(baseline_rows)
baseline_profile.head()


In [ ]:
baseline_profile.to_csv(BASE / "05_Artifacts" / "baseline_profile_generated.csv", index=False)


## 3. Compare incoming data against the baseline


In [ ]:
findings = []
for col in future_df.columns:
    if col not in train_df.columns:
        continue
    if pd.api.types.is_numeric_dtype(train_df[col]) and pd.api.types.is_numeric_dtype(future_df[col]):
        b = train_df[col].mean(); i = future_df[col].mean()
        pct = ((i-b)/b) if b else np.nan
        findings.append({"column_name": col, "check_type": "mean_shift", "pct_change": pct, "flag": abs(pct) > 0.10 if pd.notna(pct) else False})
    else:
        findings.append({"column_name": col, "check_type": "top_category_shift", "pct_change": np.nan, "flag": train_df[col].astype(str).mode()[0] != future_df[col].astype(str).mode()[0]})
findings_df = pd.DataFrame(findings)
findings_df.head(20)


In [ ]:
flagged = findings_df[findings_df["flag"] == True]
flagged.to_csv(BASE / "05_Artifacts" / "flagged_drift_findings_generated.csv", index=False)
flagged


## 4. Bias and explainability talking point
Use SageMaker Clarify or a simple segmented review to show that customer groups should be checked, not just model accuracy.


In [ ]:
segment = train_df.groupby("Contract")["label"].mean().reset_index().rename(columns={"label": "avg_churn_rate"})
segment


## 5. Optional AWS Glue bridge
Open `06_Assets/code/optional_glue_job_uc4.py` to show how monitoring profiles can be created with a batch ETL job.


## 6. Decision ladder
Minor findings keep the model active. Larger findings trigger review, retraining, or rollback.
